In [1]:
print("hello")

hello


In [2]:
import os
import ast
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, hamming_loss, f1_score
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Import ฟังก์ชันบันทึกผลการประเมินลง Markdown (.md)
import sys, gc
if '..' not in sys.path:
    sys.path.append('..')
from src.eval_logger import save_eval_report


Device: cuda


In [3]:
DATA_PATH = "../goemotions/data/go_emotions_test_with_translation.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "goemotions/data/go_emotions_test_with_translation.csv"

df = pd.read_csv(DATA_PATH).dropna(subset=["text_th"]).reset_index(drop=True)
print(f"จำนวนข้อมูลภาษาไทยที่พร้อมใช้งาน: {len(df)} แถว")

# ---------------------------------------------------------
# 📌 ส่วนที่เพิ่มเข้ามา: Label Mapping (GoEmotions 28 -> Ekman 7)
# ---------------------------------------------------------
ORIGINAL_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral"
]

EKMAN_MAPPING = {
    "Joy": ["admiration", "amusement", "approval", "caring", "desire", "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"],
    "Anger": ["anger", "annoyance", "disapproval"],
    "Sadness": ["disappointment", "embarrassment", "grief", "remorse", "sadness"],
    "Surprise": ["confusion", "curiosity", "realization", "surprise"],
    "Fear": ["fear", "nervousness"],
    "Disgust": ["disgust"],
    "Neutral": ["neutral"]
}

EKMAN_LABELS = list(EKMAN_MAPPING.keys())
NUM_LABELS = len(EKMAN_LABELS)

# สร้าง Dictionary แปลง ID เดิม เป็น ID ใหม่ (Ekman)
ekman_label_to_id = {label: i for i, label in enumerate(EKMAN_LABELS)}
orig_id_to_ekman_id = {}

for orig_id, orig_label in enumerate(ORIGINAL_LABELS):
    for ekman_label, orig_list in EKMAN_MAPPING.items():
        if orig_label in orig_list:
            orig_id_to_ekman_id[orig_id] = ekman_label_to_id[ekman_label]

def map_to_ekman(orig_labels):
    """ยุบรวมอารมณ์เดิมให้เป็น Ekman และตัดตัวซ้ำออกด้วย set()"""
    mapped = set()
    for label in orig_labels:
        if isinstance(label, int):  # ถ้าระบุเป็น Index
            if label in orig_id_to_ekman_id:
                mapped.add(orig_id_to_ekman_id[label])
        elif isinstance(label, str): # ถ้าระบุเป็น String
            for ekman_label, orig_list in EKMAN_MAPPING.items():
                if label in orig_list:
                    mapped.add(ekman_label_to_id[ekman_label])
    return list(mapped)

# แปลง String list กลับเป็น Python list และ Map เป็น Ekman
df["labels_list"] = df["labels"].apply(ast.literal_eval)
df["ekman_labels"] = df["labels_list"].apply(map_to_ekman)

# Binarize ด้วย จำนวนคลาสของ Ekman
mlb = MultiLabelBinarizer(classes=list(range(NUM_LABELS)))
Y_bin = mlb.fit_transform(df["ekman_labels"])
X_texts = df["text_th"].astype(str).tolist()
# ---------------------------------------------------------

# pos_weight สำหรับ Imbalanced Data
pos_counts = Y_bin.sum(axis=0)
neg_counts = len(df) - pos_counts
pos_weight = neg_counts / (pos_counts + 1e-6)
pos_weight_tensor = torch.FloatTensor(pos_weight).to(device)

# Train / Val / Test (80% / 10% / 10%)
X_train_val, X_test, Y_train_val, Y_test = train_test_split(X_texts, Y_bin, test_size=0.10, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_train_val, Y_train_val, test_size=0.1111, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

จำนวนข้อมูลภาษาไทยที่พร้อมใช้งาน: 5131 แถว
Train: 4104 | Val: 513 | Test: 514


In [4]:
class ThaiEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.FloatTensor(self.labels[idx])
        }

def train_model(model_name, use_fast=True, epochs=3, lr=2e-5):
    print(f"--- Fine-tuning {model_name} ---")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS, problem_type="multi_label_classification").to(device)
    train_ds = ThaiEmotionDataset(X_train, Y_train, tok)
    val_ds = ThaiEmotionDataset(X_val, Y_val, tok)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for b in train_loader:
            optimizer.zero_grad()
            out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
            loss = criterion(out.logits, b["labels"].to(device))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for b in val_loader:
                out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
                val_loss += criterion(out.logits, b["labels"].to(device)).item()
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
    
    del optimizer, train_loader, val_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return model, tok

# เทรน 3 Base Models (แก้ไขเพิ่ม model_xlmr ที่ขาดไป)
model_wangchan, tok_wangchan = train_model("airesearch/wangchanberta-base-att-spm-uncased", use_fast=False)
model_mbert, tok_mbert = train_model("bert-base-multilingual-cased", use_fast=True)
model_xlmr, tok_xlmr = train_model("xlm-roberta-base", use_fast=True)

--- Fine-tuning airesearch/wangchanberta-base-att-spm-uncased ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.1572 | Val Loss: 1.0962
Epoch 2/3 | Train Loss: 1.0465 | Val Loss: 1.0547
Epoch 3/3 | Train Loss: 1.0381 | Val Loss: 1.1794
--- Fine-tuning bert-base-multilingual-cased ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.1737 | Val Loss: 1.1383
Epoch 2/3 | Train Loss: 1.1542 | Val Loss: 1.1741
Epoch 3/3 | Train Loss: 1.1452 | Val Loss: 1.1483
--- Fine-tuning xlm-roberta-base ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Train Loss: 1.1712 | Val Loss: 1.1803
Epoch 2/3 | Train Loss: 1.1661 | Val Loss: 1.1109
Epoch 3/3 | Train Loss: 1.1262 | Val Loss: 1.1839


In [5]:
def get_probs(model, tok, texts):
    model.eval()
    ds = ThaiEmotionDataset(texts, np.zeros((len(texts), NUM_LABELS)), tok)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    probs = []
    with torch.no_grad():
        for b in loader:
            out = model(b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device))
            probs.append(torch.sigmoid(out.logits).cpu().numpy())
    return np.concatenate(probs, axis=0)

print("ดึง Probability Features...")
train_w = get_probs(model_wangchan, tok_wangchan, X_train)
train_m = get_probs(model_mbert, tok_mbert, X_train)
train_x = get_probs(model_xlmr, tok_xlmr, X_train)
X_train_meta = np.concatenate([train_w, train_m, train_x], axis=1)

test_w = get_probs(model_wangchan, tok_wangchan, X_test)
test_m = get_probs(model_mbert, tok_mbert, X_test)
test_x = get_probs(model_xlmr, tok_xlmr, X_test)
X_test_meta = np.concatenate([test_w, test_m, test_x], axis=1)

rf = MultiOutputClassifier(RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1))
rf.fit(X_train_meta, Y_train)
print("✅ เทรน Random Forest Meta-Classifier สำเร็จ!")

ดึง Probability Features...
✅ เทรน Random Forest Meta-Classifier สำเร็จ!


In [6]:
Y_pred = rf.predict(X_test_meta)
print(f"Micro F1: {f1_score(Y_test, Y_pred, average='micro'):.4f}")
print(f"Macro F1: {f1_score(Y_test, Y_pred, average='macro'):.4f}")
print(f"Hamming Loss: {hamming_loss(Y_test, Y_pred):.4f}")
print("Classification Report:")
print(classification_report(Y_test, Y_pred, target_names=EKMAN_LABELS, zero_division=0))

# 📌 เปลี่ยนชื่อ Report ไม่ให้ทับของเดิม
report_path = save_eval_report(
    y_true=Y_test,
    y_pred=Y_pred,
    target_names=EKMAN_LABELS,
    model_name="Stacked Ensemble Ekman (WangchanBERTa + mBERT + XLM-RoBERTa -> Random Forest)",
    hyperparams={
        "Base Models": "WangchanBERTa, mBERT, XLM-RoBERTa",
        "Meta Classifier": "RandomForest (n_estimators=200)",
        "Epochs": 3,
        "Batch Size": 16,
        "Learning Rate": "2e-5",
        "Mapping": "27 GoEmotions to 6 Ekman + Neutral"
    },
    notes="การประเมินจาก Notebook 04_base_models_ekman.ipynb (Label Mapping)",
    output_dir="../results" if os.path.exists("../results") else "results"
)

Micro F1: 0.4332
Macro F1: 0.2592
Hamming Loss: 0.1520
Classification Report:
              precision    recall  f1-score   support

         Joy       0.59      0.54      0.57       186
       Anger       0.45      0.18      0.25        73
     Sadness       0.56      0.15      0.23        34
    Surprise       0.40      0.27      0.32        63
        Fear       0.00      0.00      0.00        11
     Disgust       0.00      0.00      0.00        10
     Neutral       0.47      0.41      0.44       177

   micro avg       0.51      0.38      0.43       554
   macro avg       0.35      0.22      0.26       554
weighted avg       0.49      0.38      0.41       554
 samples avg       0.38      0.39      0.38       554

✅ Saved evaluation report to markdown file: ../results/eval_2026-10-06_01-02-04.md
📊 Updated experiment leaderboard: ../results/SUMMARY.md


In [7]:
# 📌 บันทึกโมเดลแยกโฟลเดอร์สำหรับ Ekman
SAVE_DIR = "saved_models_ekman"
if not os.path.exists(SAVE_DIR):
    SAVE_DIR = "../saved_models_ekman"
os.makedirs(SAVE_DIR, exist_ok=True)

model_wangchan.save_pretrained(f"{SAVE_DIR}/wangchan_model")
tok_wangchan.save_pretrained(f"{SAVE_DIR}/wangchan_tokenizer")
model_mbert.save_pretrained(f"{SAVE_DIR}/mbert_model")
tok_mbert.save_pretrained(f"{SAVE_DIR}/mbert_tokenizer")
model_xlmr.save_pretrained(f"{SAVE_DIR}/xlm_roberta_model")
tok_xlmr.save_pretrained(f"{SAVE_DIR}/xlm_roberta_tokenizer")

joblib.dump(rf, f"{SAVE_DIR}/random_forest_meta.joblib")
with open(f"{SAVE_DIR}/label_classes.json", "w", encoding="utf-8") as f:
    json.dump(EKMAN_LABELS, f, ensure_ascii=False, indent=2)
print(f"✅ บันทึกโมเดลทั้งหมดลงใน {SAVE_DIR} เรียบร้อย!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ บันทึกโมเดลทั้งหมดลงใน ../saved_models_ekman เรียบร้อย!
